In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from transformers import pipeline

# -----------------------------
# 1. Generate Synthetic Dataset
# -----------------------------
categories = ['Electronics', 'Grocery', 'Clothing', 'Furniture']
priority = ['High', 'Medium', 'Low']
carriers = ['Air', 'Road', 'Sea']
weather = ['Clear', 'Rain', 'Snow']
locations = ['Seattle', 'Chicago', 'Dallas', 'San Francisco']

data = []
for i in range(3000):
    qty = random.randint(1, 100)
    weight = round(qty * random.uniform(0.2, 2.5), 2)
    dist = random.randint(10, 2000)
    base_time = dist / random.uniform(40, 80)
    pr = random.choice(priority)
    wt = random.choice(weather)
    priority_factor = 0.8 if pr == 'High' else 1.0
    weather_factor = 1.2 if wt != 'Clear' else 1.0
    fulfillment_time = round(base_time * priority_factor * weather_factor, 2)
    data.append([f"ORD-{i+1}", random.choice(categories), qty, weight, 
                 pr, random.choice(['Credit Card','COD']), 
                 random.choice(locations), dist, random.choice(carriers),
                 wt, random.choice([0,1]),
                 random.randint(0,5), fulfillment_time])

df = pd.DataFrame(data, columns=[
    "Order_ID","Product_Category","Order_Quantity","Order_Weight",
    "Order_Priority","Payment_Method","Warehouse_Location",
    "Distance_to_Customer","Carrier_Type","Weather_Conditions",
    "Holiday_Season","Past_Delivery_Delay","Fulfillment_Time"
])

# -----------------------------
# 2. Train Model
# -----------------------------
X = df.drop(["Order_ID", "Fulfillment_Time"], axis=1)
y = df["Fulfillment_Time"]

# Encode categorical columns
le = LabelEncoder()
for col in X.select_dtypes(include=['object']).columns:
    X[col] = le.fit_transform(X[col].astype(str))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluation
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

# -----------------------------
# 3. Falcon-7B for Explanations
# -----------------------------
gen_model = pipeline("text-generation", model="tiiuae/falcon-7b-instruct", device_map="auto")

def explain_prediction(input_data, predicted_time):
    prompt = f"""
    You are an AI assistant for Costco Order Management.
    Given this order data: {input_data},
    The predicted fulfillment time is {predicted_time} hours.
    Explain to a customer why this delivery time was predicted.
    """
    result = gen_model(prompt, max_length=120, num_return_sequences=1, do_sample=True)
    return result[0]['generated_text']

# -----------------------------
# 4. Streamlit App UI
# -----------------------------
st.title("🛒 Costco Order Fulfillment Predictor")
st.write("Enter order details below to predict delivery time and get an AI-generated explanation.")

product = st.selectbox("Product Category", categories)
priority_choice = st.selectbox("Order Priority", priority)
payment = st.selectbox("Payment Method", ["Credit Card", "COD"])
warehouse = st.selectbox("Warehouse Location", locations)
carrier = st.selectbox("Carrier Type", carriers)
weather_choice = st.selectbox("Weather Conditions", weather)
qty = st.number_input("Order Quantity", min_value=1, max_value=100, value=10)
weight = st.number_input("Order Weight (kg)", min_value=1.0, value=5.0)
distance = st.slider("Distance to Customer (km)", 10, 2000, 500)
holiday = st.selectbox("Holiday Season?", [0, 1])
past_delay = st.slider("Past Delivery Delay (days)", 0, 10, 2)

if st.button("Predict Fulfillment Time"):
    # Prepare input
    input_data = {
        "Product_Category": product,
        "Order_Quantity": qty,
        "Order_Weight": weight,
        "Order_Priority": priority_choice,
        "Payment_Method": payment,
        "Warehouse_Location": warehouse,
        "Distance_to_Customer": distance,
        "Carrier_Type": carrier,
        "Weather_Conditions": weather_choice,
        "Holiday_Season": holiday,
        "Past_Delivery_Delay": past_delay
    }
    
    df_input = pd.DataFrame([input_data])
    for col in df_input.select_dtypes(include=['object']).columns:
        df_input[col] = le.fit_transform(df_input[col].astype(str))
    
    prediction = model.predict(df_input)[0]
    
    st.success(f"📦 Predicted Fulfillment Time: {round(prediction, 2)} hours")
    
    with st.spinner("AI is explaining..."):
        explanation = explain_prediction(input_data, round(prediction, 2))
    st.info(explanation)

# Show model performance
st.subheader("📊 Model Performance")
st.write(f"**MAE:** {mae:.2f}, **MSE:** {mse:.2f}, **RMSE:** {rmse:.2f}, **R²:** {r2:.2f}")


In [ ]:
from pyngrok import ngrok
!streamlit run app.py &

# Create tunnel
public_url = ngrok.connect(8501)
print("🌍 App URL:", public_url)
